## Workspace setup

In [ ]:
from datetime import datetime 
import uproot
import awkward as ak
import tensorflow as tf
import numpy as np
import importlib
from functools import partial

from tensorflow.data import Dataset, TFRecordDataset
from tensorflow.data.experimental import TFRecordWriter
from tensorflow.train import BytesList, FloatList, Int64List
from tensorflow.train import Example, Features, Feature
import tensorflow_datasets as tfds

import sys
sys.path.append("/home/akalinow/scratch/ELITPC/TPCReco/PythonAnalysis/python/")

import io_functions as io

### Load and convert SimEvent data

In [ ]:
%%time
importlib.reload(io)

dataPath = '/home/akalinow/scratch/ELITPC/TPCReco/build/resources/'
rootfiles = [dataPath+'SimEvent_Track3D_TwoProng_gun_MC.root:TPCData']
outputDir = 'SimEvent_Track3D_TwoProng_gun_MC'

# Convert ROOT files to TF format and save to output directory
io.convertROOT(rootfiles, outputDir, fields= io.simEventFields)

# Load the dataset and benchmark it
batchSize = 32
simDataset = tf.data.Dataset.load(outputDir, compression="GZIP")
simDataset = dataset.batch(batchSize).cache()

tfds.benchmark(simDataset, batch_size=batchSize)

# Second pass profiting from the cache
tfds.benchmark(simDataset, batch_size=batchSize)


************ Summary ************



  0%|          | 0/23 [00:00<?, ?it/s]

Examples/sec (First included) 337.13 ex/sec (total: 768 ex, 2.28 sec)
Examples/sec (First only) 199.79 ex/sec (total: 32 ex, 0.16 sec)
Examples/sec (First excluded) 347.52 ex/sec (total: 736 ex, 2.12 sec)

************ Summary ************



2025-06-27 15:22:11.427433: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


  0%|          | 0/23 [00:00<?, ?it/s]

Examples/sec (First included) 99129.24 ex/sec (total: 768 ex, 0.01 sec)
Examples/sec (First only) 6918.24 ex/sec (total: 32 ex, 0.00 sec)
Examples/sec (First excluded) 235745.64 ex/sec (total: 736 ex, 0.00 sec)
CPU times: user 13.6 s, sys: 2.98 s, total: 16.6 s
Wall time: 10.5 s


2025-06-27 15:22:11.438963: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


,duration,num_examples,avg
first+lasts,0.007747,768,99129.237420
first,0.004625,32,6918.241305
lasts,0.003122,736,235745.636864


### Load and convert RecoEvent data

In [ ]:
%%time
importlib.reload(io)

dataPath = '/home/akalinow/scratch/ELITPC/TPCReco/build/resources/'
rootfiles = [dataPath+'SimEvent_Track3D_TwoProng_gun_MC.root:TPCRecoData']
outputDir = 'RecoEvent_Track3D_TwoProng_gun_MC'

# Convert ROOT files to TF format and save to output directory
io.convertROOT(rootfiles, outputDir, fields=io.recoEventFields)

# Load the dataset and benchmark it
batchSize = 32
recoDataset = tf.data.Dataset.load(outputDir, compression="GZIP")
recoDataset = dataset.batch(batchSize).cache()

tfds.benchmark(recoDataset, batch_size=batchSize)

# Second pass profiting from the cache
tfds.benchmark(recoDataset, batch_size=batchSize)


************ Summary ************



  0%|          | 0/23 [00:00<?, ?it/s]

Examples/sec (First included) 332.62 ex/sec (total: 768 ex, 2.31 sec)
Examples/sec (First only) 223.89 ex/sec (total: 32 ex, 0.14 sec)
Examples/sec (First excluded) 339.79 ex/sec (total: 736 ex, 2.17 sec)

************ Summary ************



2025-06-27 15:24:11.813381: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


  0%|          | 0/23 [00:00<?, ?it/s]

Examples/sec (First included) 72421.79 ex/sec (total: 768 ex, 0.01 sec)
Examples/sec (First only) 4982.17 ex/sec (total: 32 ex, 0.01 sec)
Examples/sec (First excluded) 176007.67 ex/sec (total: 736 ex, 0.00 sec)
CPU times: user 4.21 s, sys: 818 ms, total: 5.03 s
Wall time: 3.8 s


2025-06-27 15:24:11.825545: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


,duration,num_examples,avg
first+lasts,0.010605,768,72421.791302
first,0.006423,32,4982.167732
lasts,0.004182,736,176007.667812


### Merge SimEvent and RecoEvent data


In [ ]:
# merge SimEvent and RecoEvent data
simDataset = tf.data.Dataset.load('SimEvent_Track3D_TwoProng_gun_MC', compression="GZIP")
recoDataset = tf.data.Dataset.load('RecoEvent_Track3D_TwoProng_gun_MC', compression="GZIP")

mergedDataset = tf.data.Dataset.zip((simDataset, recoDataset))
mergedDataset = mergedDataset.map(
    lambda sim, reco: {
        'sim': sim,
        'reco': reco
    }
)
# save merged dataset
outputDir = 'MergedEvent_Track3D_TwoProng_gun_MC'
mergedDataset.save(outputDir, compression="GZIP")